# DCLC Model Training — Google Colab Notebook

**Paper:** *Deep contrastive learning–based redundancy elimination for similar data in cloud storage systems* (2026)

This notebook trains the DCLC context-aware Transformer encoder using self-supervised InfoNCE contrastive learning.

### Paper Architecture & Training Hyperparameters (Section 3.2 & Section 4.1)
- **Encoder Layers:** 4
- **Attention Heads:** 8
- **Model Dimension ($d_{\text{model}}$):** 128
- **Feed-Forward Dimension ($d_{\text{ff}}$):** 512
- **Positional Encoding:** Sinusoidal
- **Activation:** GELU
- **Dropout:** 0.1
- **Output Semantic Vector:** $L_2$-normalized $\mathbb{R}^{128}$
- **Loss:** InfoNCE with temperature $\tau = 0.07$
- **Negatives:** Up to 15 boundary-aware negative samples per anchor
- **Optimizer:** AdamW with learning rate $3 \times 10^{-4}$
- **Batch Size:** 256 (adjusted automatically for small smoke-test datasets)
- **Epochs:** 20 with early-stopping patience of 5

## 1. Setup Environment and Imports

In [ ]:
import math
import json
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load and Verify the Prepared Dataset and Manifest

Loads `dclc_train_data.npz` and `dataset_manifest.json`.
- Checks `data/processed/` and the current working directory (`.` / `/content/`).
- Prompts for upload in Google Colab if either file is missing.
- Performs strict verification of fingerprint counts and ordering to prevent invalid negative sampling.

In [ ]:
def locate_or_upload_file(filename: str) -> Path:
    candidate_dirs = [Path("data/processed"), Path(".")]
    for d in candidate_dirs:
        p = d / filename
        if p.exists() and p.is_file():
            return p
    
    # If running in Google Colab, trigger file upload dialog
    try:
        from google.colab import files
        print(f"Required file '{filename}' was not found in {candidate_dirs}.")
        print(f"Please upload '{filename}' now:")
        uploaded = files.upload()
        if filename in uploaded:
            return Path(filename)
        for fn in uploaded.keys():
            if fn.endswith(Path(filename).suffix):
                return Path(fn)
    except ImportError:
        pass
        
    raise FileNotFoundError(
        f"Critical Error: '{filename}' not found and could not be loaded. "
        f"Please place '{filename}' in the current working directory."
    )

npz_path = locate_or_upload_file("dclc_train_data.npz")
manifest_path = locate_or_upload_file("dataset_manifest.json")

print(f"Loaded dataset archive from: {npz_path.resolve()}")
print(f"Loaded dataset manifest from: {manifest_path.resolve()}")

raw_archive = np.load(npz_path)
manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))

context_sequences = raw_archive["context_sequences"].astype(np.float32)  # (N, 9, 128)
initial_features = raw_archive["initial_features"].astype(np.float32)    # (N, 128)
file_ids = raw_archive["file_ids"].astype(np.int32)                      # (N,)
chunk_indices = raw_archive["chunk_indices"].astype(np.int32)            # (N,)
byte_offsets = raw_archive["byte_offsets"].astype(np.int64)              # (N,)
byte_lengths = raw_archive["byte_lengths"].astype(np.int32)              # (N,)

total_chunks = len(context_sequences)
manifest_chunks = manifest_data.get("chunks", [])

# Strict verification: NEVER replace missing fingerprints with dummy placeholders
if len(manifest_chunks) != total_chunks:
    raise ValueError(
        f"Integrity check failed: manifest contains {len(manifest_chunks)} chunk entries, "
        f"but dataset archive contains {total_chunks} context sequences. Training halted."
    )

fingerprints = []
for i, chunk_entry in enumerate(manifest_chunks):
    # Ensure chunk ordering matches exactly across both files
    if int(chunk_entry["file_id"]) != int(file_ids[i]) or int(chunk_entry["chunk_index"]) != int(chunk_indices[i]):
        raise ValueError(
            f"Integrity check failed: Chunk ordering mismatch at index {i}. "
            f"Manifest has (file_id={chunk_entry.get('file_id')}, chunk_index={chunk_entry.get('chunk_index')}), "
            f"Archive has (file_id={file_ids[i]}, chunk_index={chunk_indices[i]}). Training halted."
        )
    fingerprints.append(chunk_entry["sha256"])

print(f"Verified dataset successfully:")
print(f"  Total chunks: {total_chunks}")
print(f"  Context sequences shape: {context_sequences.shape}, dtype: {context_sequences.dtype}")
print(f"  Initial features shape: {initial_features.shape}, dtype: {initial_features.dtype}")
print(f"  Fingerprint count: {len(fingerprints)} (verified exact 1-to-1 alignment)")

## 3. DCLC Transformer Model Definition

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 128, max_len: int = 128, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)


class DCLCTransformerContextEncoder(nn.Module):
    def __init__(
        self,
        d_model: int = 128,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
        output_dim: int = 128,
        normalize_output: bool = True,
    ):
        super().__init__()
        self.d_model = d_model
        self.output_dim = output_dim
        self.normalize_output = normalize_output

        self.pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.projection_head = nn.Linear(d_model, output_dim)

    def forward(self, x: torch.Tensor, center_idx: int = None) -> torch.Tensor:
        if center_idx is None:
            center_idx = x.size(1) // 2
        x = self.pos_encoder(x)
        encoded = self.transformer_encoder(x)
        center_repr = encoded[:, center_idx, :]
        out = self.projection_head(center_repr)
        if self.normalize_output:
            out = F.normalize(out, p=2, dim=-1)
        return out

## 4. InfoNCE Loss and Negative Sampling Definition

### Zero-Negative Handling Rules
- If an anchor in a batch has **zero valid negative samples**, it is **excluded from the loss average** rather than contributing an artificial 0.0000 loss.
- If an entire batch contains **zero usable anchors**, the batch update is safely skipped with a warning.
- This prevents artificial loss drops and avoids NaN/infinite losses.

In [ ]:
class InfoNCELoss(nn.Module):
    def __init__(self, temperature: float = 0.07, reduction: str = "mean"):
        super().__init__()
        self.temperature = temperature
        self.reduction = reduction

    def forward(self, anchor: torch.Tensor, positive: torch.Tensor, negatives: torch.Tensor, negatives_mask: torch.Tensor = None) -> torch.Tensor:
        b, d = anchor.shape
        b_neg, k_neg, d_neg = negatives.shape

        pos_sim = torch.sum(anchor * positive, dim=-1, keepdim=True) / self.temperature
        neg_sim = torch.bmm(negatives, anchor.unsqueeze(-1)).squeeze(-1) / self.temperature

        if negatives_mask is not None:
            neg_sim = neg_sim.masked_fill(~negatives_mask, -1e9)
            has_valid_negatives = negatives_mask.sum(dim=-1) > 0
        else:
            has_valid_negatives = (
                torch.ones(b, dtype=torch.bool, device=anchor.device)
                if k_neg > 0
                else torch.zeros(b, dtype=torch.bool, device=anchor.device)
            )

        logits = torch.cat([pos_sim, neg_sim], dim=1)
        targets = torch.zeros(b, dtype=torch.long, device=anchor.device)
        loss_per_sample = F.cross_entropy(logits, targets, reduction="none")

        num_usable = has_valid_negatives.sum()
        if num_usable == 0:
            if self.reduction == "none":
                return torch.zeros(b, device=anchor.device)
            return torch.tensor(0.0, device=anchor.device, requires_grad=True)

        if self.reduction == "mean":
            return (loss_per_sample * has_valid_negatives.float()).sum() / num_usable.float()
        elif self.reduction == "sum":
            return (loss_per_sample * has_valid_negatives.float()).sum()
        else:
            return loss_per_sample * has_valid_negatives.float()


class BoundaryAwareNegativeSampler:
    def __init__(self, file_ids, chunk_indices, fingerprints, initial_features, context_radius=4, proximity_threshold=0.85, max_negatives=15, seed=42):
        self.num_chunks = len(file_ids)
        self.file_ids = np.asarray(file_ids, dtype=np.int32)
        self.chunk_indices = np.asarray(chunk_indices, dtype=np.int32)
        self.fingerprints = list(fingerprints)
        norms = np.linalg.norm(initial_features, axis=-1, keepdims=True)
        norms[norms == 0] = 1.0
        self.normalized_feats = (initial_features / norms).astype(np.float32)
        self.context_radius = context_radius
        self.proximity_threshold = proximity_threshold
        self.max_negatives = max_negatives
        self.rng = np.random.default_rng(seed)

    def get_valid_negative_indices(self, anchor_idx: int) -> np.ndarray:
        candidate_mask = np.ones(self.num_chunks, dtype=bool)
        candidate_mask[anchor_idx] = False
        is_same_file = self.file_ids == self.file_ids[anchor_idx]
        is_in_window = is_same_file & (np.abs(self.chunk_indices - self.chunk_indices[anchor_idx]) <= self.context_radius) & candidate_mask
        is_same_fp = np.array([fp == self.fingerprints[anchor_idx] for fp in self.fingerprints], dtype=bool) & candidate_mask
        sims = self.normalized_feats @ self.normalized_feats[anchor_idx]
        is_proximate = (sims >= self.proximity_threshold) & candidate_mask
        is_invalid = (~candidate_mask) | is_in_window | is_same_fp | is_proximate
        valid_indices = np.where(~is_invalid)[0]
        if len(valid_indices) == 0:
            return np.empty(0, dtype=np.int32)
        if len(valid_indices) > self.max_negatives:
            return self.rng.choice(valid_indices, size=self.max_negatives, replace=False).astype(np.int32)
        return valid_indices.astype(np.int32)

## 5. Dataset Loader Setup

In [ ]:
sampler = BoundaryAwareNegativeSampler(
    file_ids=file_ids,
    chunk_indices=chunk_indices,
    fingerprints=fingerprints,
    initial_features=initial_features,
    context_radius=4,
    proximity_threshold=0.85,
    max_negatives=15,
)

intended_batch_size = 256
effective_batch_size = min(intended_batch_size, max(4, total_chunks // 2))
print(f"Configured batch size: {effective_batch_size} (intended full size: {intended_batch_size})")

class DCLCDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        return torch.from_numpy(self.sequences[idx]).float(), idx

train_dataset = DCLCDataset(context_sequences)
train_loader = DataLoader(train_dataset, batch_size=effective_batch_size, shuffle=True)

model = DCLCTransformerContextEncoder().to(device)
loss_fn = InfoNCELoss(temperature=0.07)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

print("Model parameter count:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Training Loop (20 Epochs with Early Stopping Patience 5)

### Important Research & Methodological Notes

#### 1. Positive Pair Formulation (Provisional Engineering Choice)
The DCLC paper (Section 3.2, Page 9) states: *"For each central block $C_i$, semantically enhanced neighbors are selected as the positive sample $h_i^+$."*
Because the paper does not specify the exact algorithmic formula for selecting or constructing these neighbors, this implementation uses **stochastic forward-pass dropout augmentation** (passing the same context sequence twice through the model with active dropout $p=0.1$, following standard self-supervised practice like SimCSE). **This is an engineering placeholder and should not be claimed as an exact reproduction of the authors' method.**

#### 2. Negative Stop-Gradient Behavior (`torch.no_grad()`)
In the loop below, negative sample representations are computed inside `torch.no_grad()`:
```python
with torch.no_grad():
    neg_vectors = model(neg_seqs)
```
- **Why chosen:** Backpropagating through 15 negative sequences for every anchor would require 16× more memory per step, causing GPU/CPU Out-Of-Memory (OOM).
- **Gradient Flow Consequence:** Gradients flow strictly through the anchor and positive embeddings. Negatives act as fixed target representations during the step (similar to MoCo / BYOL / SimSiam stop-gradient branches) rather than updating simultaneously.

#### 3. Early Stopping Clarification
Early stopping in this notebook monitors the training loss on the small 51-chunk sample dataset. **This is strictly a smoke-test mechanism** to verify convergence and prevent infinite loops. It does **not** prove general resemblance detection or high compression ratio on real workloads.

In [ ]:
num_epochs = 20
patience = 5
best_loss = float("inf")
patience_counter = 0

Path("checkpoints").mkdir(parents=True, exist_ok=True)
checkpoint_path = Path("checkpoints/dclc_encoder.pt")

model.train()
print("Starting DCLC contrastive representation training...")

for epoch in range(1, num_epochs + 1):
    epoch_losses = []
    batches_trained = 0
    
    for batch_seqs, batch_indices in train_loader:
        batch_seqs = batch_seqs.to(device)
        optimizer.zero_grad()

        # 1. Forward pass for anchors h_i
        anchor_emb = model(batch_seqs)  # (B, 128)

        # 2. Forward pass for positive views h_i^+ (via independent dropout stochastic pass)
        pos_emb = model(batch_seqs)     # (B, 128)

        # 3. Gather negatives for each anchor in batch using BoundaryAwareNegativeSampler
        batch_size_cur = len(batch_indices)
        neg_embs_list = []
        masks_list = []

        for b_i in range(batch_size_cur):
            anchor_idx = int(batch_indices[b_i])
            neg_indices = sampler.get_valid_negative_indices(anchor_idx)
            
            if len(neg_indices) > 0:
                neg_seqs = torch.from_numpy(context_sequences[neg_indices]).float().to(device)
                # Stop-gradient on negatives to avoid 16x memory footprint
                with torch.no_grad():
                    neg_vectors = model(neg_seqs)  # (K_valid, 128)
                
                # Pad to 15 if fewer negatives available
                if len(neg_indices) < 15:
                    pad_len = 15 - len(neg_indices)
                    pad = torch.zeros(pad_len, 128, device=device)
                    neg_vectors = torch.cat([neg_vectors, pad], dim=0)
                    mask = torch.cat([torch.ones(len(neg_indices), dtype=torch.bool), torch.zeros(pad_len, dtype=torch.bool)])
                else:
                    mask = torch.ones(15, dtype=torch.bool)
            else:
                # Zero valid negatives: mask out completely so it doesn't affect loss
                neg_vectors = torch.zeros(15, 128, device=device)
                mask = torch.zeros(15, dtype=torch.bool)

            neg_embs_list.append(neg_vectors)
            masks_list.append(mask)

        batch_negatives = torch.stack(neg_embs_list).to(device)  # (B, 15, 128)
        batch_masks = torch.stack(masks_list).to(device)         # (B, 15)

        # Check if entire batch has zero valid negative samples
        total_valid_negs = batch_masks.sum().item()
        if total_valid_negs == 0:
            print(f"Warning: Batch skipped because no anchors have valid negative samples.")
            continue

        loss = loss_fn(anchor_emb, pos_emb, batch_negatives, negatives_mask=batch_masks)
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
        batches_trained += 1

    if batches_trained == 0:
        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] - Warning: All batches skipped (no valid negatives).")
        continue

    avg_loss = sum(epoch_losses) / len(epoch_losses)
    print(f"Epoch [{epoch:02d}/{num_epochs:02d}] - Loss: {avg_loss:.4f} (trained on {batches_trained} batches)")

    # Early stopping
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch} epochs (patience = {patience}).")
            break

print(f"Training finished! Best Loss: {best_loss:.4f}")
print(f"Model checkpoint saved to: {checkpoint_path}")

## 7. Download Trained Weights

Use the cell below if you are running in Google Colab to download the trained checkpoint `checkpoints/dclc_encoder.pt` to your local machine.

In [ ]:
try:
    from google.colab import files
    files.download(str(checkpoint_path))
    print("Download initiated.")
except ImportError:
    print(f"Local environment: checkpoint available at {checkpoint_path.resolve()}")